[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SETU-AIML-2026/DH-Infra-AI/blob/main/topic-05-sprint-4/unit-1-week-10-model-serving/notebook-1-week-10-model-serving-lab/Lab_10_Model_Serving.ipynb)


# Lab 10: From a trained model to an inference API

**Sprint 4 · Data Handling and Infrastructure for AI · SETU Certificate in AI/ML**

In Sprint 3 you trained the Titanic neural network (Lab 7), tracked experiments with MLflow (Lab 8) and ran a distributed training job on Vertex AI that wrote `gs://YOUR_BUCKET/model/model.pt` (Lab 9).

A file of weights cannot answer a question. In this lab you turn the Sprint 3 model into a **running prediction service** that another program can call over HTTP.

### What you will build

```text
JSON request ─► validate (Pydantic) ─► preprocess (train-fitted) ─► PyTorch forward pass ─► sigmoid ─► JSON response
                                                                                                   (+ model version)
```

### Learning outcomes
By the end of this lab you will be able to:
1. **Audit** a model artifact before serving it, and explain why weights alone are not a usable model.
2. Package a **model bundle**: weights, a train-fitted preprocessing contract and versioned metadata.
3. Implement a **FastAPI** inference service that validates requests, loads the model once and returns versioned predictions.
4. Prove **training–serving parity** with automated tests.
5. Prepare **evidence for criterion 4.1** of the final report (inference service architecture and implementation).

### Practicalities
* **Runtime:** a CPU runtime is enough. Unlike Labs 7–9 you do **not** need a T4 GPU.
* **Cost:** the core lab uses no paid cloud resources. The optional Cloud Run section at the end does.
* **Time:** about 2 hours. Work through the parts in order; every cell is meant to be run.
* **Data:** either your own GCS bucket from Lab 3 (`sharded/train.csv`, `dev.csv`, `test.csv`) or the public Titanic CSV, re-split exactly as in Lab 3.

> ✅ **Checkpoints** tell you what you should see. 📝 **Your turn** marks questions to answer in the notebook. Keep your answers: they are evidence for the final report.

## 0. Setup

Colab already ships PyTorch, pandas and scikit-learn. We add the serving stack (FastAPI, Uvicorn), an HTTP client for testing, and MLflow.

Remember Tamas's note in Lab 8: *dependencies should be pinned*. We install here, then in Part 8 we write the **exact versions you actually used** into `requirements.txt`.

In [ ]:
%pip install -q fastapi uvicorn httpx requests mlflow pytest

In [ ]:
import hashlib, json, math, os, platform, shutil, subprocess, sys, time, warnings
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Python ", platform.python_version())
for pkg in ["torch", "pandas", "numpy", "scikit-learn", "fastapi", "uvicorn", "pydantic", "mlflow"]:
    print(f"{pkg:<13}", version(pkg))
print("Running in Colab:", IN_COLAB)

### Configuration

Edit this cell only.

* `DATA_SOURCE = "gcs"` reads the splits from **your** bucket (same files as Labs 7–9). `"public"` downloads the raw Titanic CSV and recreates the Lab 3 split with the same seed and stratification, so you get the same rows.
* The hyperparameters default to the Lab 9 values. If your Lab 8 grid search found a better configuration, **use it** and say so in your report.

In [ ]:
# ---- EDIT THESE VALUES -------------------------------------------------------
DATA_SOURCE = "public"                  # "gcs" or "public"
PROJECT_ID  = "YOUR_GOOGLE_CLOUD_PROJECT_ID"   # only needed for "gcs"
BUCKET_NAME = "YOUR_EXISTING_GCS_BUCKET"       # only needed for "gcs"
AUDIT_MY_LAB9_MODEL = True              # if using "gcs": also download and audit gs://BUCKET/model/model.pt

MODEL_NAME    = "titanic-survival-mlp"
MODEL_VERSION = "1.0.0"                 # bump this whenever weights OR preprocessing change

# Lab 9 defaults (replace with your best Lab 8 configuration if you have one)
HIDDEN_SIZE   = 1000
EPOCHS        = 100
BATCH_SIZE    = 64
LEARNING_RATE = 0.001
SEED          = 42
THRESHOLD     = 0.5
# -----------------------------------------------------------------------------

RAW_URL   = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
WORK_DIR  = Path("lab10")
SERVE_DIR = WORK_DIR / "serving"          # everything you would put in a Git repository
BUNDLE_DIR = SERVE_DIR / "model_bundle"   # the deployable model artifact
for d in [SERVE_DIR / "app", SERVE_DIR / "tests" / "fixtures", BUNDLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    torch.manual_seed(seed); np.random.seed(seed)
set_seed()
print("Working directory:", WORK_DIR.resolve())

## Part 1: Load exactly the same data as Sprint 3

An inference service is only meaningful if we know **which data** the model learned from. We reuse the train/dev/test split from Lab 3 and keep the **test split locked**: we only touch train (fit everything) and dev (choose and check). The test set is for the final evaluation in your report.

In [ ]:
if DATA_SOURCE == "gcs":
    assert BUCKET_NAME != "YOUR_EXISTING_GCS_BUCKET", "Set BUCKET_NAME first."
    if IN_COLAB:
        from google.colab import auth
        auth.authenticate_user()
    base = f"gs://{BUCKET_NAME}/sharded"
    df_train = pd.read_csv(f"{base}/train.csv")
    df_dev   = pd.read_csv(f"{base}/dev.csv")
    df_test  = pd.read_csv(f"{base}/test.csv")
    DATA_URI = base
else:
    # Same procedure, seed and stratification as the Lab 3 GCS notebook
    df_raw = pd.read_csv(RAW_URL)
    df_train, df_tmp = train_test_split(df_raw, test_size=0.20, random_state=42, stratify=df_raw["Survived"])
    df_dev, df_test  = train_test_split(df_tmp, test_size=0.50, random_state=42, stratify=df_tmp["Survived"])
    DATA_URI = RAW_URL + " (Lab 3 split, random_state=42, stratified 80/10/10)"

# Guard against leakage: the three splits must not share passengers
ids = [set(d["PassengerId"]) for d in (df_train, df_dev, df_test)]
assert not (ids[0] & ids[1] or ids[0] & ids[2] or ids[1] & ids[2]), "Splits overlap!"

print(f"train {df_train.shape}  dev {df_dev.shape}  test {df_test.shape}")
print("\nMissing values in the TRAIN split (raw features):")
print(df_train[["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex", "Embarked"]].isna().sum().to_string())

✅ **Checkpoint:** 712 / 89 / 90 rows, and **137 missing `Age` values** plus 2 missing `Embarked` values in train. Those missing values are the heart of the next part.

## Part 2: Audit the Sprint 3 artifact *before* you serve it

The first rule of serving: **never put an artifact behind an API without checking it.** A model file can load without error and still be useless.

We write the model definition and an audit function into `serving/app/model.py`. The *same file* will later be imported by the API, so the notebook and the service cannot drift apart.

The architecture is copied from Lab 9: three hidden layers with sigmoid activations and a final `Linear` layer that outputs a **logit**. Lab 9 trained it with `BCEWithLogitsLoss`, so the service must apply `sigmoid` **exactly once**. (Labs 7 and 8 put a final `Sigmoid` *inside* the network and used `BCELoss`. Those weights are not interchangeable with this class.)

In [ ]:
%%writefile lab10/serving/app/model.py
"""Model definition shared by training (notebook) and serving (API).

Identical to Sprint 3 Lab 9: the final Linear layer returns a LOGIT.
Apply torch.sigmoid exactly once, at inference time.
"""
import torch
from torch import nn

ARCHITECTURE_ID = "titanic-mlp-3xsigmoid-logit-v1"
EXPECTED_KEYS = [f"network.{i}.{p}" for i in (0, 2, 4, 6) for p in ("weight", "bias")]


class TitanicMLP(nn.Module):
    def __init__(self, input_size: int = 8, hidden_size: int = 1000):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size), nn.Sigmoid(),
            nn.Linear(hidden_size, hidden_size), nn.Sigmoid(),
            nn.Linear(hidden_size, hidden_size), nn.Sigmoid(),
            nn.Linear(hidden_size, 1),            # logit, no final sigmoid
        )

    def forward(self, x):
        return self.network(x)


def audit_state_dict(state: dict, input_size: int = 8) -> dict:
    """Checks a state_dict BEFORE it is served. Returns a report; report['ok'] is False on any problem."""
    problems = []
    keys = list(state.keys())
    if sorted(keys) != sorted(EXPECTED_KEYS):
        problems.append(f"unexpected keys: {keys} (a Lab 7/8 model with a final Sigmoid has the same keys; check the metadata)")
        return {"ok": False, "problems": problems}
    hidden = int(state["network.0.weight"].shape[0])
    if state["network.0.weight"].shape[1] != input_size:
        problems.append(f"input size {state['network.0.weight'].shape[1]} != {input_size}")
    if tuple(state["network.6.weight"].shape) != (1, hidden):
        problems.append(f"output layer shape {tuple(state['network.6.weight'].shape)} != (1, {hidden})")
    n_params = sum(t.numel() for t in state.values())
    n_nonfinite = int(sum((~torch.isfinite(t)).sum().item() for t in state.values()))
    if n_nonfinite:
        problems.append(f"{n_nonfinite:,} of {n_params:,} parameters are NaN/Inf: the model cannot produce valid predictions")
    return {"ok": not problems, "hidden_size": hidden, "n_params": n_params,
            "n_nonfinite": n_nonfinite, "problems": problems}

In [ ]:
sys.path.insert(0, str(SERVE_DIR.resolve()))
from app.model import TitanicMLP, audit_state_dict, ARCHITECTURE_ID

lab9_report = None
if DATA_SOURCE == "gcs" and AUDIT_MY_LAB9_MODEL:
    from google.cloud import storage
    blob = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME).blob("model/model.pt")
    if blob.exists():
        blob.download_to_filename(WORK_DIR / "lab9_model.pt")
        lab9_state = torch.load(WORK_DIR / "lab9_model.pt", map_location="cpu", weights_only=True)
        lab9_report = audit_state_dict(lab9_state)
        print("Audit of YOUR gs://.../model/model.pt:", json.dumps(lab9_report, indent=2))
    else:
        print("No model/model.pt in your bucket (Lab 9 not run). That is fine: continue.")
else:
    print("Skipping the download. The next cell reproduces what Lab 9 would have produced.")

### What did Lab 9 actually save?

The next cell copies the Lab 9 preprocessing function **verbatim** and runs one epoch of the Lab 9 training loop on the real training split. Watch the loss.

In [ ]:
def lab9_preprocess_split(df_split, df_train):          # copied verbatim from Lab 9 src/train.py
    features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex', 'Embarked']
    df_processed = df_split[features].copy()
    df_processed['Age'].fillna(df_train['Age'].median())
    df_processed['Fare'].fillna(df_train['Fare'].median())
    df_processed['Embarked'].fillna(df_train['Embarked'].mode()[0])
    df_processed = pd.get_dummies(df_processed, columns=['Sex', 'Embarked'], drop_first=True)
    expected_columns = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']
    for col in expected_columns:
        if col not in df_processed.columns:
            df_processed[col] = 0
    return df_processed[expected_columns]

X9 = StandardScaler().fit_transform(lab9_preprocess_split(df_train, df_train)).astype(np.float32)
print("NaN values reaching the network:", int(np.isnan(X9).sum()))

set_seed()
m9 = TitanicMLP(8, HIDDEN_SIZE)
opt9, loss9 = torch.optim.Adam(m9.parameters(), lr=LEARNING_RATE), nn.BCEWithLogitsLoss()
y9 = torch.tensor(df_train["Survived"].values, dtype=torch.float32).unsqueeze(1)
for i, (xb, yb) in enumerate(torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.from_numpy(X9), y9), batch_size=BATCH_SIZE, shuffle=True)):
    opt9.zero_grad(); loss = loss9(m9(xb), yb); loss.backward(); opt9.step()
    if i < 3: print(f"batch {i}: loss = {loss.item()}")

print("\nAudit of the reproduced Lab 9 weights:")
print(json.dumps(audit_state_dict(m9.state_dict()), indent=2))

✅ **Checkpoint:** the loss is `nan` from the first batch, and the audit reports that **every parameter is NaN**.

**Why?** `df['Age'].fillna(value)` *returns a new Series*; it does not change `df`. Lab 9 never assigns the result back, so the 137 missing ages travel through `StandardScaler` (which passes NaN through) into the network. One NaN in a batch makes the loss NaN, the gradients NaN, and after one optimiser step every weight is NaN. The "validation accuracy" printed by such a job is **0.618**: `NaN > 0.5` is `False`, so the model predicts *did not survive* for everyone, which happens to be the majority class.

Labs 7 and 8 use `fillna(..., inplace=True)` on a column. That works with pandas 2.x but **silently does nothing with pandas 3** (Copy-on-Write). The same notebook can therefore produce a different model next year.

This is not a "gotcha"; it is the most common way real ML services fail. Nothing crashed. The job reported success, the file uploaded, the accuracy looked plausible. Only an **artifact audit** catches it.

The lessons we carry into serving:
1. Preprocessing must be **code with tests**, not a few notebook lines.
2. The service must **refuse to start** if the artifact fails the audit.
3. "It loads" is not "it works". Compare against a **baseline** (here: 0.618 = always predicting the majority class).

📝 **Your turn (Q1):** If you audited your own Lab 9 model above, paste the report. Was it valid? If you ran Labs 7/8 in Colab, check `pd.__version__` there: which pandas behaviour did you get?

## Part 3: A preprocessing contract that training and serving share

The service receives **raw** passenger fields (`"Sex": "female"`), but the network expects **eight standardised numbers in a fixed order**. Everything between the two must be:

* **fitted on the training split only** (no leakage: the medians, mode, mean and standard deviation come from train);
* **stored** with the model (the service must not recompute anything from requests);
* **row-independent**: one passenger must be encoded the same way alone or in a batch.

That last point rules out `pd.get_dummies` at serving time. Watch what it does to a single request:

In [ ]:
one_request = pd.DataFrame([{"Pclass": 3, "Age": 28, "SibSp": 0, "Parch": 0, "Fare": 7.9, "Sex": "female", "Embarked": "Q"}])
print("Training columns :", ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S'])
print("get_dummies(1 row):", list(pd.get_dummies(one_request, columns=["Sex", "Embarked"], drop_first=True).columns))

With one row, `drop_first=True` drops the *only* category present, so both `Sex` and `Embarked` vanish. The Lab 9 "fill missing columns with 0" step would then silently encode this woman from Queenstown as a **woman from Cherbourg**. No error, just a wrong prediction.

So we write an explicit, deterministic encoder. It needs only NumPy (no pandas or scikit-learn in the service), and it saves itself to JSON.

In [ ]:
%%writefile lab10/serving/app/preprocessing.py
"""Train-fitted preprocessing contract, shared by training and serving.

Raw fields  : Pclass, Age, SibSp, Parch, Fare, Sex, Embarked
Model input : Pclass, Age, SibSp, Parch, Fare, Sex_male, Embarked_Q, Embarked_S  (standardised)
All statistics are fitted on the TRAINING split only and stored in preprocessing.json.
"""
import json
import math
from collections import Counter
from pathlib import Path

import numpy as np

RAW_FIELDS = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex", "Embarked"]
FEATURE_ORDER = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex_male", "Embarked_Q", "Embarked_S"]
SEX_VALUES = ("female", "male")
EMBARKED_VALUES = ("C", "Q", "S")


def _missing(v) -> bool:
    return v is None or (isinstance(v, float) and math.isnan(v))


def _records(data):
    """Accepts a list of dicts or a pandas DataFrame (pandas is not required)."""
    if hasattr(data, "to_dict"):
        data = data[RAW_FIELDS].to_dict(orient="records")
    return list(data)


class TitanicPreprocessor:
    VERSION = "titanic-preprocessing-v1"

    def __init__(self, impute: dict, mean: list, scale: list, feature_order=FEATURE_ORDER):
        self.impute = impute
        self.mean = np.asarray(mean, dtype=np.float32)
        self.scale = np.asarray(scale, dtype=np.float32)
        self.feature_order = list(feature_order)

    # ---- fitting (training time only) -------------------------------------
    @classmethod
    def fit(cls, train_data):
        rows = _records(train_data)
        ages = [float(r["Age"]) for r in rows if not _missing(r["Age"])]
        fares = [float(r["Fare"]) for r in rows if not _missing(r["Fare"])]
        counts = Counter(r["Embarked"] for r in rows if not _missing(r["Embarked"]))
        top = max(counts.values())
        impute = {"Age": float(np.median(ages)), "Fare": float(np.median(fares)),
                  "Embarked": sorted(k for k, c in counts.items() if c == top)[0]}
        proto = cls(impute, np.zeros(8), np.ones(8))
        X = proto.encode(rows).astype(np.float64)
        std = X.std(axis=0)                     # population std, as sklearn's StandardScaler
        std[std == 0] = 1.0
        return cls(impute, X.mean(axis=0), std)

    # ---- transforming (training AND serving) ------------------------------
    def encode_one(self, r: dict) -> list:
        sex, emb = r["Sex"], r.get("Embarked")
        emb = self.impute["Embarked"] if _missing(emb) else emb
        if sex not in SEX_VALUES:
            raise ValueError(f"unknown Sex: {sex!r}")
        if emb not in EMBARKED_VALUES:
            raise ValueError(f"unknown Embarked: {emb!r}")
        age = self.impute["Age"] if _missing(r.get("Age")) else float(r["Age"])
        fare = self.impute["Fare"] if _missing(r.get("Fare")) else float(r["Fare"])
        return [float(r["Pclass"]), age, float(r["SibSp"]), float(r["Parch"]), fare,
                1.0 if sex == "male" else 0.0, 1.0 if emb == "Q" else 0.0, 1.0 if emb == "S" else 0.0]

    def encode(self, data) -> np.ndarray:
        return np.asarray([self.encode_one(r) for r in _records(data)], dtype=np.float32)

    def transform(self, data) -> np.ndarray:
        return ((self.encode(data) - self.mean) / self.scale).astype(np.float32)

    # ---- persistence -------------------------------------------------------
    def to_dict(self) -> dict:
        return {"version": self.VERSION, "raw_fields": RAW_FIELDS, "feature_order": self.feature_order,
                "impute": self.impute, "mean": self.mean.tolist(), "scale": self.scale.tolist(),
                "categories": {"Sex": list(SEX_VALUES), "Embarked": list(EMBARKED_VALUES)}}

    def save(self, path):
        Path(path).write_text(json.dumps(self.to_dict(), indent=2))

    @classmethod
    def load(cls, path):
        d = json.loads(Path(path).read_text())
        if d.get("version") != cls.VERSION:
            raise ValueError(f"preprocessing version {d.get('version')} != {cls.VERSION}")
        return cls(d["impute"], d["mean"], d["scale"], d["feature_order"])

In [ ]:
import importlib, app.preprocessing as pp
importlib.reload(pp)
from app.preprocessing import TitanicPreprocessor, FEATURE_ORDER

pre = TitanicPreprocessor.fit(df_train)          # fitted on TRAIN only
print("Imputation values (from train):", pre.impute)

# Independent reference: a corrected pandas + scikit-learn pipeline (fillna ASSIGNED back)
def reference_pipeline(df, train):
    d = df[["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex", "Embarked"]].copy()
    d["Age"] = d["Age"].fillna(train["Age"].median())
    d["Fare"] = d["Fare"].fillna(train["Fare"].median())
    d["Embarked"] = d["Embarked"].fillna(train["Embarked"].mode()[0])
    d["Sex_male"] = (d["Sex"] == "male").astype(float)
    d["Embarked_Q"] = (d["Embarked"] == "Q").astype(float)
    d["Embarked_S"] = (d["Embarked"] == "S").astype(float)
    return d[FEATURE_ORDER].astype(float)

ref_scaler = StandardScaler().fit(reference_pipeline(df_train, df_train))
for name, d in [("train", df_train), ("dev", df_dev)]:
    ours, ref = pre.transform(d), ref_scaler.transform(reference_pipeline(d, df_train))
    print(f"{name}: max |ours - sklearn| = {np.abs(ours - ref).max():.2e}   NaN: {int(np.isnan(ours).sum())}")

# Row independence: one passenger alone == the same passenger inside a batch
row = df_dev.iloc[[5]]
assert np.allclose(pre.transform(row), pre.transform(df_dev)[5]), "encoding depends on the batch!"
print("Row-independent encoding: OK")

✅ **Checkpoint:** differences below `1e-5` (float32 rounding), **zero NaN**, and *Row-independent encoding: OK*. Imputation: Age 28.5, Fare 14.4542, Embarked "S".

## Part 4: Retrain the Lab 9 architecture with the fixed pipeline, and track it

We now train the **same network and hyperparameters as Lab 9**, on CPU, with the corrected preprocessing. It takes 10–60 seconds. Unlike Lab 8 we log the **dev loss per epoch** as well as the training loss, so the learning curves can show over- or underfitting.

MLflow gives us **provenance**: the run ID goes into the model's metadata, so any prediction can be traced back to the exact run, parameters and data. We use a local SQLite tracking store; point `mlflow.set_tracking_uri` at a remote server (Databricks, DagsHub) if you set one up in Lab 8.

In [ ]:
import mlflow
mlflow.set_tracking_uri(f"sqlite:///{(WORK_DIR / 'mlflow.db').resolve()}")
mlflow.set_experiment("Titanic_Serving_Lab10")

X_tr, y_tr = pre.transform(df_train), df_train["Survived"].values.astype(np.float32)
X_dv, y_dv = pre.transform(df_dev),   df_dev["Survived"].values.astype(np.float32)

def train_model(hidden_size=HIDDEN_SIZE, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, seed=SEED):
    set_seed(seed)
    model = TitanicMLP(8, hidden_size)
    opt, crit = torch.optim.Adam(model.parameters(), lr=lr), nn.BCEWithLogitsLoss()
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr).unsqueeze(1)),
        batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(seed))
    Xd, yd = torch.from_numpy(X_dv), torch.from_numpy(y_dv).unsqueeze(1)
    for epoch in range(epochs):
        model.train(); total = 0.0
        for xb, yb in loader:
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
            total += loss.item() * len(xb)
        model.eval()
        with torch.no_grad():
            dev_logits = model(Xd)
            dev_loss = crit(dev_logits, yd).item()
            dev_acc = ((torch.sigmoid(dev_logits) >= THRESHOLD).float() == yd).float().mean().item()
        mlflow.log_metrics({"train_loss": total / len(X_tr), "dev_loss": dev_loss, "dev_accuracy_epoch": dev_acc}, step=epoch)
    return model

t0 = time.time()
with mlflow.start_run(run_name=f"{MODEL_NAME}-{MODEL_VERSION}") as run:
    mlflow.log_params({"architecture": ARCHITECTURE_ID, "hidden_size": HIDDEN_SIZE, "epochs": EPOCHS,
                       "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "seed": SEED,
                       "threshold": THRESHOLD, "preprocessing": TitanicPreprocessor.VERSION, "data": DATA_URI})
    model = train_model()
    train_seconds = time.time() - t0

    model.eval()
    with torch.no_grad():
        p_dev = torch.sigmoid(model(torch.from_numpy(X_dv))).squeeze(1).numpy()
    yhat = (p_dev >= THRESHOLD).astype(int)
    dev_metrics = {
        "dev_accuracy": accuracy_score(y_dv, yhat), "dev_precision": precision_score(y_dv, yhat),
        "dev_recall": recall_score(y_dv, yhat), "dev_f1": f1_score(y_dv, yhat),
        "dev_roc_auc": roc_auc_score(y_dv, p_dev),
        "majority_baseline_accuracy": float(max(y_dv.mean(), 1 - y_dv.mean())),
        "train_seconds": train_seconds}
    mlflow.log_metrics(dev_metrics)
    RUN_ID = run.info.run_id

print(f"Trained in {train_seconds:.1f}s  |  MLflow run {RUN_ID}")
print(json.dumps({k: round(v, 4) for k, v in dev_metrics.items()}, indent=2))
print("Confusion matrix (dev) [[TN FP] [FN TP]]:\n", confusion_matrix(y_dv, yhat))

✅ **Checkpoint:** dev accuracy around **0.84** (the exact value can vary slightly between machines) against a majority baseline of **0.618**. The audit of these weights must pass:

In [ ]:
report = audit_state_dict(model.state_dict())
print(json.dumps(report, indent=2))
assert report["ok"], "Do not continue: this model must not be served."

In [ ]:
import matplotlib.pyplot as plt
client = mlflow.tracking.MlflowClient()
fig, ax = plt.subplots(figsize=(8, 4))
for key in ["train_loss", "dev_loss"]:
    hist = client.get_metric_history(RUN_ID, key)
    ax.plot([h.step for h in hist], [h.value for h in hist], label=key)
ax.set_xlabel("epoch"); ax.set_ylabel("BCE loss"); ax.set_title("Learning curves (from MLflow)"); ax.legend(); ax.grid(alpha=.3)
plt.show()

📝 **Your turn (Q2):** Does the dev loss show under- or overfitting? Would you serve the model from the last epoch, or should training have stopped earlier (Lecture 7, early stopping)?

## Part 5: Package a model bundle

A deployable model is **three files that travel together**:

| File | Contents | Why the service needs it |
|---|---|---|
| `model.pt` | `state_dict` (weights only, loaded with `weights_only=True`) | the learned parameters |
| `preprocessing.json` | imputation values, mean, scale, category lists, feature order | to turn raw JSON into the exact 8 inputs the weights expect |
| `metadata.json` | name, version, architecture, `hidden_size`, threshold, metrics, data source, MLflow run ID, library versions, **SHA-256 of `model.pt`** | to rebuild the network, verify integrity and make every prediction traceable |

`model.pt` alone answers none of: *which architecture? which hidden size? which preprocessing? is the file intact? which run produced it?*

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

torch.save(model.state_dict(), BUNDLE_DIR / "model.pt")
pre.save(BUNDLE_DIR / "preprocessing.json")

metadata = {
    "model_name": MODEL_NAME,
    "model_version": MODEL_VERSION,
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "model": {"architecture": ARCHITECTURE_ID, "framework": "pytorch", "input_size": 8,
              "hidden_size": HIDDEN_SIZE, "output": "logit (apply sigmoid once)",
              "feature_order": FEATURE_ORDER, "decision_threshold": THRESHOLD,
              "positive_class": "survived"},
    "training": {"data": DATA_URI, "train_rows": int(len(df_train)), "dev_rows": int(len(df_dev)),
                 "hyperparameters": {"epochs": EPOCHS, "batch_size": BATCH_SIZE,
                                     "learning_rate": LEARNING_RATE, "seed": SEED},
                 "mlflow_run_id": RUN_ID, "preprocessing_version": TitanicPreprocessor.VERSION},
    "evaluation": {"split": "dev", **{k: round(float(v), 4) for k, v in dev_metrics.items()}},
    "artifact": {"model_file": "model.pt", "model_sha256": sha256(BUNDLE_DIR / "model.pt")},
    "environment": {"python": platform.python_version(), "torch": version("torch"), "numpy": version("numpy")},
    "intended_use": "Teaching example: Titanic survival probability for a single passenger record.",
    "limitations": "Historical 1912 data (891 passengers); not suitable for decisions about real people.",
}
(BUNDLE_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_artifacts(str(BUNDLE_DIR), artifact_path="model_bundle")

for f in sorted(BUNDLE_DIR.iterdir()):
    print(f"{f.name:<20} {f.stat().st_size/1024:8.1f} KB")

**Reload test:** load the bundle from disk exactly as the service will, and check it reproduces the in-memory predictions.

In [ ]:
reloaded = TitanicMLP(8, metadata["model"]["hidden_size"])
reloaded.load_state_dict(torch.load(BUNDLE_DIR / "model.pt", map_location="cpu", weights_only=True), strict=True)
reloaded.eval()
pre2 = TitanicPreprocessor.load(BUNDLE_DIR / "preprocessing.json")
with torch.inference_mode():
    p_reload = torch.sigmoid(reloaded(torch.from_numpy(pre2.transform(df_dev)))).squeeze(1).numpy()
print("max |in-memory - reloaded| =", float(np.abs(p_reload - p_dev).max()))
assert np.allclose(p_reload, p_dev, atol=1e-6)

## Part 6: Write the inference service

Four small modules, each with one job:

| Module | Responsibility |
|---|---|
| `schemas.py` | the **public contract**: which fields, types and ranges are accepted, and what comes back |
| `preprocessing.py` | raw fields → 8 standardised features (already written, shared with training) |
| `model.py` | architecture + artifact audit (already written, shared with training) |
| `main.py` | HTTP: load the bundle **once** at startup, expose endpoints, handle errors, measure time |

In [ ]:
%%writefile lab10/serving/app/schemas.py
"""Public request/response contract of the inference API (Pydantic v2)."""
from typing import Literal, Optional

from pydantic import BaseModel, ConfigDict, Field

EXAMPLE = {"Pclass": 3, "Sex": "female", "Age": 28, "SibSp": 0, "Parch": 0, "Fare": 7.75, "Embarked": "Q"}


class Passenger(BaseModel):
    # extra="forbid": a misspelt or unexpected field (e.g. "Survived") is an error, not silently ignored
    model_config = ConfigDict(extra="forbid", json_schema_extra={"examples": [EXAMPLE]})

    Pclass: Literal[1, 2, 3] = Field(description="Ticket class: 1, 2 or 3")
    Sex: Literal["male", "female"]
    Age: Optional[float] = Field(default=None, ge=0, le=100, description="Years; null = training-set median")
    SibSp: int = Field(ge=0, le=10, description="Siblings/spouses aboard")
    Parch: int = Field(ge=0, le=10, description="Parents/children aboard")
    Fare: Optional[float] = Field(default=None, ge=0, le=600, description="Ticket fare; null = training-set median")
    Embarked: Optional[Literal["C", "Q", "S"]] = Field(default=None, description="Port; null = training-set mode")


class Prediction(BaseModel):
    survival_probability: float = Field(ge=0, le=1)
    predicted_class: Literal[0, 1]
    predicted_label: Literal["survived", "did_not_survive"]
    threshold: float
    model_version: str


class BatchRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    passengers: list[Passenger] = Field(min_length=1, max_length=100)


class BatchPrediction(BaseModel):
    model_version: str
    count: int
    predictions: list[Prediction]

In [ ]:
%%writefile lab10/serving/app/main.py
"""Titanic survival inference API.

Run locally:   uvicorn app.main:app --port 8000      (from the serving/ directory)
Docs:          http://127.0.0.1:8000/docs
"""
import hashlib
import json
import logging
import os
import time
from contextlib import asynccontextmanager
from pathlib import Path

import torch
from fastapi import FastAPI, HTTPException, Request

from .model import ARCHITECTURE_ID, TitanicMLP, audit_state_dict
from .preprocessing import TitanicPreprocessor
from .schemas import BatchPrediction, BatchRequest, Passenger, Prediction

MODEL_DIR = Path(os.getenv("MODEL_DIR", Path(__file__).resolve().parent.parent / "model_bundle"))
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")
log = logging.getLogger("titanic-api")
STATE: dict = {}


def _sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def load_bundle(model_dir: Path):
    """Loads and verifies the bundle. Any problem raises: the service refuses to start."""
    metadata = json.loads((model_dir / "metadata.json").read_text())
    if metadata["model"]["architecture"] != ARCHITECTURE_ID:
        raise RuntimeError(f"architecture {metadata['model']['architecture']} is not supported")
    if _sha256(model_dir / "model.pt") != metadata["artifact"]["model_sha256"]:
        raise RuntimeError("model.pt does not match the checksum in metadata.json")
    state = torch.load(model_dir / "model.pt", map_location="cpu", weights_only=True)
    report = audit_state_dict(state, metadata["model"]["input_size"])
    if not report["ok"]:
        raise RuntimeError(f"model failed the artifact audit: {report['problems']}")
    if report["hidden_size"] != metadata["model"]["hidden_size"]:
        raise RuntimeError("hidden_size in metadata does not match the weights")
    model = TitanicMLP(metadata["model"]["input_size"], report["hidden_size"])
    model.load_state_dict(state, strict=True)
    model.eval()                                  # inference behaviour (matters for dropout/batch-norm)
    pre = TitanicPreprocessor.load(model_dir / "preprocessing.json")
    if pre.feature_order != metadata["model"]["feature_order"]:
        raise RuntimeError("feature order in preprocessing.json does not match metadata.json")
    return model, pre, metadata


@asynccontextmanager
async def lifespan(app: FastAPI):
    torch.set_num_threads(int(os.getenv("TORCH_NUM_THREADS", "1")))
    model, pre, metadata = load_bundle(MODEL_DIR)          # ONCE, not per request
    STATE.update(model=model, pre=pre, metadata=metadata)
    log.info("loaded %s v%s from %s", metadata["model_name"], metadata["model_version"], MODEL_DIR)
    yield
    STATE.clear()


app = FastAPI(title="Titanic Survival Inference API", version="1.0.0", lifespan=lifespan,
              description="Sprint 4 Lab 10: serves the Sprint 3 Titanic MLP.")


@app.middleware("http")
async def add_timing(request: Request, call_next):
    start = time.perf_counter()
    response = await call_next(request)
    ms = (time.perf_counter() - start) * 1000
    response.headers["X-Process-Time-ms"] = f"{ms:.2f}"
    if "metadata" in STATE:
        response.headers["X-Model-Version"] = STATE["metadata"]["model_version"]
    # Log the route, status and latency, but NOT the passenger data (data minimisation)
    log.info("%s %s -> %s in %.2f ms", request.method, request.url.path, response.status_code, ms)
    return response


def _predict(records: list[dict]) -> list[Prediction]:
    meta = STATE["metadata"]
    threshold = meta["model"]["decision_threshold"]
    try:
        x = torch.from_numpy(STATE["pre"].transform(records))
    except (KeyError, ValueError) as exc:
        raise HTTPException(status_code=422, detail=str(exc)) from exc
    with torch.inference_mode():                           # no autograd bookkeeping
        probs = torch.sigmoid(STATE["model"](x)).squeeze(1).tolist()   # sigmoid exactly once
    return [Prediction(survival_probability=round(p, 6), predicted_class=int(p >= threshold),
                       predicted_label="survived" if p >= threshold else "did_not_survive",
                       threshold=threshold, model_version=meta["model_version"]) for p in probs]


@app.get("/health")
def health():
    """Liveness/readiness: true only once a verified model is in memory."""
    return {"status": "ok" if "model" in STATE else "loading", "model_loaded": "model" in STATE}


@app.get("/model")
def model_info():
    """What is being served: version, provenance and offline evaluation (no secrets)."""
    m = STATE["metadata"]
    return {k: m[k] for k in ("model_name", "model_version", "created_at", "model", "training", "evaluation", "intended_use", "limitations")}


@app.post("/predict", response_model=Prediction)
def predict(passenger: Passenger):
    return _predict([passenger.model_dump()])[0]


@app.post("/predict/batch", response_model=BatchPrediction)
def predict_batch(batch: BatchRequest):
    preds = _predict([p.model_dump() for p in batch.passengers])
    return BatchPrediction(model_version=STATE["metadata"]["model_version"], count=len(preds), predictions=preds)

In [ ]:
Path(SERVE_DIR / "app" / "__init__.py").write_text("")
print(*sorted(str(p.relative_to(WORK_DIR)) for p in SERVE_DIR.rglob("*") if p.is_file() and "__pycache__" not in str(p)), sep="\n")

## Part 7: Run the service and call it

### 7a. In-process (always works)
FastAPI's `TestClient` calls the app without opening a network port. It runs the full stack: routing, validation, startup, middleware.

In [ ]:
for mod in [m for m in list(sys.modules) if m == "app" or m.startswith("app.")]:
    del sys.modules[mod]                          # pick up the files we just wrote
os.environ["MODEL_DIR"] = str(BUNDLE_DIR.resolve())
from fastapi.testclient import TestClient
from app.main import app

passenger = {"Pclass": 3, "Sex": "female", "Age": 28, "SibSp": 0, "Parch": 0, "Fare": 7.75, "Embarked": "Q"}
with TestClient(app) as client:
    print("GET /health ->", client.get("/health").json())
    r = client.post("/predict", json=passenger)
    print("POST /predict ->", r.status_code, r.json())
    print("headers       ->", {k: v for k, v in r.headers.items() if k.startswith("x-")})

### 7b. As a real server

Now start Uvicorn as a **separate process**, exactly as it would run in a container, and talk to it over HTTP.

In [ ]:
import requests
server_log = open(WORK_DIR / "server.log", "w")
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
                          cwd=SERVE_DIR, env={**os.environ, "MODEL_DIR": str(BUNDLE_DIR.resolve())},
                          stdout=server_log, stderr=subprocess.STDOUT)
BASE = "http://127.0.0.1:8000"
for _ in range(60):
    try:
        if requests.get(f"{BASE}/health", timeout=1).json().get("model_loaded"):
            break
    except requests.ConnectionError:
        time.sleep(0.5)
print("Server PID", server.pid, "->", requests.get(f"{BASE}/health").json())

In [ ]:
!curl -s -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" -d '{"Pclass":1,"Sex":"male","Age":45,"SibSp":1,"Parch":0,"Fare":83.5,"Embarked":"C"}'

In [ ]:
print(json.dumps(requests.get(f"{BASE}/model").json()["evaluation"], indent=2))

**Interactive docs.** FastAPI generates OpenAPI documentation from `schemas.py`. In Colab the next cell opens it in a new window; locally, browse to <http://127.0.0.1:8000/docs>. Use **Try it out** on `/predict`. A screenshot of this page is good evidence for your report.

In [ ]:
if IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_window(8000, path="/docs")
else:
    print("Open http://127.0.0.1:8000/docs in your browser")

### 7c. The contract in action: invalid requests

A good API rejects input it was not designed for, with a **clear 422 error**, instead of inventing a number.

In [ ]:
cases = {
    "valid":                       passenger,
    "unknown category Sex='other'": {**passenger, "Sex": "other"},
    "negative age":                {**passenger, "Age": -3},
    "missing required Pclass":     {k: v for k, v in passenger.items() if k != "Pclass"},
    "unexpected field 'Survived'": {**passenger, "Survived": 1},
    "Age = null (imputed)":        {**passenger, "Age": None},
}
for name, body in cases.items():
    r = requests.post(f"{BASE}/predict", json=body)
    detail = r.json()["detail"][0]["msg"] if r.status_code == 422 else r.json()["survival_probability"]
    print(f"{name:<32} -> {r.status_code}  {detail}")

📝 **Your turn (Q3):** `Age: null` is accepted and imputed with the training median, but `Sex: "other"` is rejected. Argue for or against each design choice. What would you change if this API were used for real people?

## Part 8: Automated tests, including training–serving parity

The most valuable test for a serving system is **parity**: the API must return the same probabilities the notebook computed when it evaluated the model. If preprocessing drifts (a different median, a swapped column, a double sigmoid), parity fails even though every other test passes.

We save 20 dev passengers with their offline probabilities as a test fixture, then write a `pytest` suite.

In [ ]:
fixture = []
for i in range(20):
    raw = df_dev.iloc[i][["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].to_dict()
    raw = {k: (None if isinstance(v, float) and math.isnan(v) else (int(v) if k in ("Pclass", "SibSp", "Parch") else v)) for k, v in raw.items()}
    fixture.append({"request": raw, "expected_probability": float(p_dev[i])})
(SERVE_DIR / "tests" / "fixtures" / "dev_parity.json").write_text(json.dumps(fixture, indent=2))
print(f"{len(fixture)} parity cases saved; {sum(c['request']['Age'] is None for c in fixture)} of them have Age = null")

In [ ]:
%%writefile lab10/serving/tests/test_api.py
import json
import os
from pathlib import Path

import pytest
from fastapi.testclient import TestClient

HERE = Path(__file__).parent
os.environ.setdefault("MODEL_DIR", str(HERE.parent / "model_bundle"))
from app.main import app  # noqa: E402

VALID = {"Pclass": 3, "Sex": "female", "Age": 28, "SibSp": 0, "Parch": 0, "Fare": 7.75, "Embarked": "Q"}


@pytest.fixture(scope="module")
def client():
    with TestClient(app) as c:      # runs startup (bundle load + audit)
        yield c


def test_health(client):
    assert client.get("/health").json() == {"status": "ok", "model_loaded": True}


def test_prediction_contract(client):
    r = client.post("/predict", json=VALID)
    assert r.status_code == 200
    body = r.json()
    assert 0.0 <= body["survival_probability"] <= 1.0
    assert body["predicted_class"] == int(body["survival_probability"] >= body["threshold"])
    assert body["model_version"] == r.headers["x-model-version"]


@pytest.mark.parametrize("change", [{"Sex": "other"}, {"Embarked": "X"}, {"Pclass": 4},
                                    {"Age": -1}, {"Fare": -5}, {"Survived": 1}])
def test_invalid_input_is_rejected(client, change):
    assert client.post("/predict", json={**VALID, **change}).status_code == 422


def test_missing_required_field(client):
    body = {k: v for k, v in VALID.items() if k != "Sex"}
    assert client.post("/predict", json=body).status_code == 422


def test_optional_fields_are_imputed(client):
    r = client.post("/predict", json={**VALID, "Age": None, "Fare": None, "Embarked": None})
    assert r.status_code == 200


def test_repeatable(client):
    a = client.post("/predict", json=VALID).json()
    b = client.post("/predict", json=VALID).json()
    assert a == b


def test_batch_equals_single(client):
    others = [VALID, {**VALID, "Sex": "male", "Pclass": 1}]
    batch = client.post("/predict/batch", json={"passengers": others}).json()
    singles = [client.post("/predict", json=p).json() for p in others]
    assert batch["count"] == 2 and batch["predictions"] == singles


def test_batch_limit(client):
    assert client.post("/predict/batch", json={"passengers": [VALID] * 101}).status_code == 422


def test_training_serving_parity(client):
    cases = json.loads((HERE / "fixtures" / "dev_parity.json").read_text())
    for case in cases:
        got = client.post("/predict", json=case["request"]).json()["survival_probability"]
        assert got == pytest.approx(case["expected_probability"], abs=1e-5), case["request"]

In [ ]:
!cd lab10/serving && {sys.executable} -m pytest -q tests 2>&1 | tail -n 15

✅ **Checkpoint:** **14 passed** (each parametrised invalid-input case counts separately). If parity fails, your service is not serving the model you evaluated.

📝 **Your turn (Q4):** Break something on purpose and re-run the tests. For example, in `preprocessing.py` swap `1.0 if sex == "male" else 0.0`, or apply `torch.sigmoid` twice in `main.py`. Which tests catch it? (Re-run the `%%writefile` cells afterwards to restore the files.) Then name two things these tests **cannot** tell you (hint: fairness, calibration, behaviour under load).

## Part 9: A first latency baseline (preview of Lecture 11)

Before talking about scaling we need a **baseline**: how long does one request take on one CPU process? Report percentiles, not just the mean. **p95** is the latency 95% of requests beat.

In [ ]:
session = requests.Session()
lat = []
for i in range(300):
    body = fixture[i % len(fixture)]["request"]
    t = time.perf_counter(); session.post(f"{BASE}/predict", json=body); lat.append((time.perf_counter() - t) * 1000)
lat = np.array(lat[20:])   # drop warm-up
print(f"sequential requests: {len(lat)}  p50 {np.percentile(lat, 50):.2f} ms  p95 {np.percentile(lat, 95):.2f} ms  max {lat.max():.2f} ms  ~{1000/lat.mean():.0f} req/s")

📝 **Your turn (Q5):** This is **one client, one server process, one request at a time**. Write down two reasons the numbers will change with 50 concurrent users. Lecture 11 measures exactly that.

## Part 10: Make it reproducible and portable

Write the **exact** versions you used into `requirements.txt` and add a `Dockerfile` and a `README`. The service needs only five libraries: no pandas, no scikit-learn, no MLflow at serving time. A smaller image starts faster and has fewer vulnerabilities.

Colab cannot run Docker, so you will not build the image here. It is the starting point for Lecture 11 and for the optional Cloud Run deployment below.

In [ ]:
def pin(pkg):
    return f"{pkg}=={version(pkg).split('+')[0]}"      # drop local tags such as +cu121

serving_reqs = [pin(p) for p in ["fastapi", "uvicorn", "pydantic", "numpy", "torch"]]
(SERVE_DIR / "requirements.txt").write_text("\n".join(serving_reqs) + "\n")
(SERVE_DIR / "requirements-dev.txt").write_text("-r requirements.txt\n" + "\n".join(pin(p) for p in ["pytest", "httpx"]) + "\n")

(SERVE_DIR / "Dockerfile").write_text('''FROM python:3.11-slim
WORKDIR /srv
# CPU-only PyTorch wheels keep the image small (no CUDA libraries needed for inference)
COPY requirements.txt .
RUN pip install --no-cache-dir --extra-index-url https://download.pytorch.org/whl/cpu -r requirements.txt
COPY app/ app/
COPY model_bundle/ model_bundle/
ENV MODEL_DIR=/srv/model_bundle TORCH_NUM_THREADS=1 PORT=8080
EXPOSE 8080
# Cloud Run injects PORT; default 8080
CMD ["sh", "-c", "uvicorn app.main:app --host 0.0.0.0 --port ${PORT}"]
''')
(SERVE_DIR / ".dockerignore").write_text("__pycache__/\n*.pyc\ntests/\n.pytest_cache/\n")

(SERVE_DIR / "README.md").write_text(f'''# Titanic survival inference API

Serves `{MODEL_NAME}` version `{MODEL_VERSION}` (MLflow run `{RUN_ID}`).

## Run locally
```bash
pip install -r requirements-dev.txt
uvicorn app.main:app --port 8000        # docs at http://127.0.0.1:8000/docs
pytest -q tests                          # contract + parity tests
```

## Run in Docker
```bash
docker build -t titanic-api:{MODEL_VERSION} .
docker run -p 8080:8080 titanic-api:{MODEL_VERSION}
```

## Endpoints
| Method | Path | Purpose |
|---|---|---|
| GET  | /health        | liveness/readiness |
| GET  | /model         | served version, provenance, offline metrics |
| POST | /predict       | one passenger |
| POST | /predict/batch | up to 100 passengers |

## Model bundle
`model_bundle/model.pt` (state_dict), `preprocessing.json` (train-fitted), `metadata.json` (version, SHA-256, metrics).
The service refuses to start if the checksum or the artifact audit fails.
''')
print((SERVE_DIR / "requirements.txt").read_text())

In [ ]:
archive = shutil.make_archive(str(WORK_DIR / "lab10_serving"), "zip", root_dir=SERVE_DIR)
print("Packaged:", archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)

Put the contents of `lab10_serving.zip` into your **project GitHub repository** (for example under `serving/`). The final report requires the inference service code in the repository.

### Optional extension: deploy to Cloud Run (uses billable resources)
Only if you have permission to spend credits. Cloud Run builds the `Dockerfile` with Cloud Build and runs it as a managed HTTPS service that scales to zero.
```bash
cd lab10/serving
gcloud run deploy titanic-api --source . --region europe-west1 \
    --memory 1Gi --cpu 1 --max-instances 2 --no-allow-unauthenticated
# ...test it, take your screenshots, then DELETE it:
gcloud run services delete titanic-api --region europe-west1
```
`--no-allow-unauthenticated` keeps the endpoint private. Call it with `curl -H "Authorization: Bearer $(gcloud auth print-identity-token)" ...`.

## Part 11: Shut down

In [ ]:
server.terminate(); server.wait(timeout=10); server_log.close()
print("Server stopped. Last log lines:")
print("".join(open(WORK_DIR / "server.log").readlines()[-3:]))

## Reflection and future work for the final report (Self Study)

Answer Q1–Q5 above, then these:

1. Why is `model.pt` alone not enough to serve the model? List everything the service needed besides the weights.
2. Where exactly is `sigmoid` applied, and what would happen to predictions if a Lab 8 model (final `Sigmoid` inside the network) were loaded into this service?
3. Why must the imputation values and scaler statistics come from the **training** split, and never from the request or the dev/test data?
4. The service refuses to start when the audit fails. Is "fail at startup" better than "start and return errors"? Consider who notices, and when.
5. How does the response make a prediction **auditable**? What else would you log in production, and what should you deliberately **not** log?

### Evidence checklist (criterion 4.1, *Inference service architecture and implementation*)
- [ ] Architecture diagram: request → validation → preprocessing → model → response, plus where the bundle comes from
- [ ] Link to the service code in your GitHub repository (`app/`, `tests/`, `Dockerfile`, `requirements.txt`, `README.md`)
- [ ] Screenshot of `/docs` and of a valid and an invalid request with their responses
- [ ] The artifact audit result and the parity test output
- [ ] Your justification: why FastAPI (or another option), why these validation rules, how the model is loaded and versioned
- [ ] Limitations you are aware of

This lab also feeds **3.7** (reproducibility and model documentation: your `metadata.json` is the start of a model card) and prepares **4.2** (scalability, Lecture 11) and **4.3–4.5** (retraining, versioning and rollback, Lecture 12).